In [ ]:
import sys ; sys.argv = [''] # This must be the first line of the cell

import numpy as np
import option
import template
import torch
import utility
import matplotlib.pyplot as plt

def get_args(parser, arglist):
    args = parser.parse_args(arglist)
    template.set_template(args)

    args.scale = list(map(lambda x: int(x), args.scale.split('+')))
    args.input_range = [float(x.strip('\"\'')) for x in args.input_range.split(',')]
    args.tensor_range = [float(x.strip('\"\'')) for x in args.tensor_range.split(',')]
    args.data_train = args.data_train.split('+')
    args.data_test = args.data_test.split('+')

    if args.epochs == 0:
        args.epochs = 1e8

    for arg in vars(args):
        if vars(args)[arg] == 'True':
            vars(args)[arg] = True
        elif vars(args)[arg] == 'False':
            vars(args)[arg] = False
            
    return args

%matplotlib widget

In [1]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(np.zeros((1000,900)))
plt.show()

In [ ]:
argdict = {
    'SCALE' : 2,
    'TEMPLATE' : 'EDSR_paper',
    'DATASET' : 'DIV2K_TIFF'
}

#--test_only
argstring = \
"""
--template {TEMPLATE}
--scale {SCALE}
--shift_mean False
--dir_data ../../Data 
--data_train {DATASET}
--data_test {DATASET}
--data_range 1-800/1-10
--input_range '-1.,1.'
--tensor_range '-1.,1.'
--n_colors 1
--no_augment
--ext img""".format(**argdict)

arglist = [i for a in argstring.split('\n') for i in a.split()]
sys.argv = arglist

args = get_args(option.parser, arglist)

---

### Exploring Loss

In [ ]:
import loss
import imageio

checkpoint = utility.checkpoint(args)
edsr_loss = loss.Loss(args, checkpoint)
edsr_loss.start_log()
edsr_loss.eval()

In [ ]:
# Computes loss of a 1000 random windows
shape = (1000, 192, 192, 1)
dtype = torch.float32

img1 = (torch.rand(shape, dtype=dtype) * 2) - 1 
img2 = (torch.rand(shape, dtype=dtype) * 2) - 1 

l = edsr_loss(img1, img2)

print(l)

In [ ]:
dtype = torch.float32

hr = imageio.imread('../../Data/DIV2K_TIFF/DIV2K_valid_HR/0801.tiff')
sr = imageio.imread('../experiment/edsrX2_DIV2K_TIFF/results-DIV2K_TIFF/0801_x2_SR.tiff')
hr, sr = [torch.tensor(img, dtype=dtype) for img in [hr, sr]]

l = edsr_loss(hr, sr)

print(l)

In [ ]:
hr_4D = hr.view(1, 1, *hr.shape)
scale = 2
scale_factor = [1, 1/scale, 1/scale, 1]
scale_factor = 1/scale

lr_torch = torch.nn.functional.interpolate(hr_4D, scale_factor=scale_factor, mode='bicubic',
                                           recompute_scale_factor=True,
                                           align_corners=False, antialias=True)
lr_sklearn = imageio.imread('../../Data/DIV2K_TIFF/DIV2K_valid_LR_bicubic/X2/0801x2.tiff')

diff = (lr_torch.squeeze() - lr_sklearn).abs()

print(diff.mean(), diff.max())

In [ ]:
plt.figure()
plt.imshow(diff.numpy(), cmap='gist_heat')
plt.grid(False)
plt.show()

---

In [ ]:
from data import div2k_tiff

In [ ]:
import data

data_loader = data.Data(args)

In [ ]:

checkpoint = utility.checkpoint(args)
_model = model.Model(args, checkpoint)


In [ ]:
img = data_loader[0]

In [ ]:
lr = img[0]

In [ ]:
lr.shape

In [ ]:
import numpy as np

imgfile = '../../Data/DIV2K_TIFF/DIV2K_train_LR_bicubic/X2/0001x2.tiff'
device = torch.device('cuda')

img = imageio.imread(imgfile)
img = np.expand_dims(img, axis=2)
lr = torch.tensor(img, device=device)

In [ ]:
_model.eval()
sr = _model(lr,0)

In [ ]:
help(_model.eval)

In [ ]:
d = data_loader.loader_test[0]

In [ ]:
d.dataset.set_scale(2)

In [ ]:
for img in d:
    print(img)